## Importing Modules

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

## Reading the Dataset

In [ ]:
path ='../input/housesalesprediction/kc_house_data.csv'
df =pd.read_csv(path)

## DATA CLEANSING AND ANALYSIS

In [ ]:
#show the first 5 rows of df
df.head()

In [ ]:

print("our dataset has {num_columns} columns and {num_rows} rows".format(num_columns = df.shape[1]
                                                                         ,num_rows = df.shape[0]))

In [ ]:
df.columns

##### Column defintions:
- id - Unique ID for each home sold
- date - Date of the home sale
- price - Price of each home sold
- bedrooms - Number of bedrooms
- bathrooms - Number of bathrooms, where .5 accounts for a room with a toilet but no shower
- sqft_living - Square footage of the apartments interior living space
- sqft_lot - Square footage of the land space
- floors - Number of floors
- waterfront - A dummy variable for whether the apartment was overlooking the waterfront or not
- view - An index from 0 to 4 of how good the view of the property was 0 = No view, 1 = Fair 2 = Average, 3 = Good, 4 = Excellent
- condition - An index from 1 to 5 on the condition of the apartment,1 = Poor- Worn out, 2 = Fair- Badly worn, 3 = Average, 4 = Good, 5= Very Good
- grade - An index from 1 to 13, where 1-3 falls short of building construction and design, 7 has an average level of construction and design, and 11-13 have a high quality level of construction and design.
- sqft_above - The square footage of the interior housing space that is above ground level
- sqft_basement - The square footage of the interior housing space that is below ground level
- yr_built - The year the house was initially built
- yr_renovated - The year of the house’s last renovation
- zipcode - What zipcode area the house is in
- lat - Lattitude
- long - Longitude
- sqft_living15 - The square footage of interior housing living space for the nearest 15 neighbors
- sqft_lot15 - The square footage of the land lots of the nearest 15 neighbors

In [ ]:
df.nunique()

In [ ]:
df.info()

In [ ]:
df.dtypes

##### date columns change type to datetime

In [ ]:
df["date"]= pd.to_datetime(df.date)

#### The age of the house:
The new column that shows the age of the house at the time of sale, and as we know, the age of the house can have a great impact on the price of the house. This column is obtained from the difference between the year of construction and the year of sale of the house.

In [ ]:
## df['date'][1].year - df.yr_built[1]       
        
age_of_house = [df['date'][index].year - df['yr_built'][index] for index in range(df.shape[0])]

df["house_age"] = age_of_house

#### view of the age of houses
- The minimum age of the house
- The most old house
- The average age of the houses

In [ ]:
df['house_age'].agg({'min','max','mean'})

#### Note:
- The minimum age is equal to -1, which has two cases, either the house has been pre-sold or an error was made during the information registration.

 Let's check this together 

In [ ]:
df[df.house_age < 0]

It is very likely that these data are noise, their number is much less than the total number of data, which means that either this data was recorded incorrectly or that such an event is very rare.

In [ ]:
df.drop(df[df.house_age < 0].index , inplace =True)
df.reset_index(inplace = True , drop =True)

#### How long has it been since the last rebuild?
We should check this as well, I think it is very important

In [ ]:
(df.yr_renovated.value_counts(normalize = True)* 100).head()

95.76 The data in this column is equal to zero, which means two things: either the information about the last renovation is not available or the house has never been renovated. I will delete this column because it does not give us useful information. But before doing this, we must also check the extent of its impact on the price of the house

In [ ]:
print("correlation between yr_renovated and price is {}".format(df['yr_renovated'].corr(df['price'])))
plt.scatter(x = df.yr_renovated ,y = df.price)
plt.title("Year Renovated with Price")
plt.xlabel("Year Renovated")
plt.ylabel("Price")
plt.show()

In [ ]:
df.drop("yr_renovated",axis = 1 , inplace =True)

#### Let's check the amount of NAN values in the dataset

In [ ]:
df.isna().sum()

In [ ]:
df.describe().T

In [ ]:
df.drop("id", axis =1 ,inplace =True)

In [ ]:
plt.figure(figsize =(10, 5))
plt.title('Price Distribuition')
plt.hist(df.price , bins =150 , color = "g" ,density =True)
plt.hist(df.price , bins =150 , color = "r" ,density =True , histtype='step' )
plt.show()

In [ ]:
sns.set(rc ={"figure.figsize":(10,5)})
sns.scatterplot(data =df ,x = 'house_age' , y = 'price'
                , hue = "floors")

In [ ]:
one = plt.hist(df[(df.floors == 1) | (df.floors ==1.5) ].price ,bins =45,fc=[1,0,0,0.5] ,label = "one floor")
two = plt.hist(df[(df.floors == 2) | (df.floors == 2.5)].price ,bins =45,fc=[0.5,0,0,1] ,label = "two floors")
three = plt.hist(df[(df.floors ==3) | (df.floors == 3.5)].price ,bins =45,fc=[0,0.5,1,0.7] ,label = "three floor")
plt.legend()

In [ ]:
df.floors.value_counts(normalize =True)

In [ ]:
plt.pie(df.floors.value_counts(normalize =True) , explode =[0.2,0,0,0,0,1] ,labels =df.floors.value_counts().index,
        autopct ="%.2f%%"
       )
plt.show()

In [ ]:
df.columns

In [ ]:
df.groupby("waterfront").price.agg({"min","max","mean"})

In [ ]:
sns.violinplot(data =df,x = "waterfront" ,y ="price")

In [ ]:
sns.violinplot(data =df,x = "waterfront" ,y ="price" , hue ="view")

In [ ]:
plt.figure(figsize =(10 , 7))

plt.subplot(2,1,1)
sns.scatterplot(data =df,x = 'sqft_living',y= 'price', hue ="view")

plt.subplot(2,1,2)
sns.scatterplot(data =df,x = 'sqft_lot',y= 'price', hue ="view")

From the graph above, it can be concluded that the better the house has a view, the more expensive it is, and also the area of the house has a positive correlation with its price, and these two features can greatly help machine learning models to predict house prices.

In [ ]:
sns.jointplot(x= df.sqft_living,y =  df.price, 
              alpha = 0.5)
plt.xlabel('Sqft-Living')
plt.ylabel('Price')
plt.show()

let's Check the correlation between zip code and house price, if it is weak, we will remove it

In [ ]:
if df.zipcode.corr(df.price) < 0.5 or df.zipcode.corr(df.price) > -0.5:
        print(df.zipcode.corr(df.price))
        df.drop("zipcode" , axis =1 , inplace =True)

In [ ]:
sns.pairplot(data = df)

In [ ]:
### corrolation plot
plt.figure(figsize =(15,7))
sns.heatmap(df.corr() , annot =True , linewidth =0.2)

#### Noise data detection using Tukey's method

In [ ]:
sns.boxplot(x ='bedrooms' , y='price' , data =df )

In [ ]:
plt.boxplot(df.price)
plt.show()

In [ ]:
def tukey(data):
    q1  , q3 = np.percentile(data, [25 ,75])
    
    iqr = q3 - q1
    lower = q1 - (iqr * 1.5)
    upper = q3 + (iqr * 1.5)
    
    return np.where((data > upper) | (data < lower))

In [ ]:
noise_index =  tukey(df.price)[0]
    

In [ ]:
df.drop(noise_index , inplace =True)

In [ ]:
df.drop("date",axis =1 , inplace =True)

In [ ]:
df.head()